# FLUX SAE Steering Notebook

This notebook demonstrates SAE (Sparse Autoencoder) steering for FLUX models, including:
- Sparse map generation and visualization
- Text stream analysis (DiT and MMDiT)
- Feature steering with multiple methods
- Empty-prompt intervention testing


In [ ]:
# Import utility functions
import sys
sys.path.append('./scripts')
from flux_sae_utils import add_feature_on_area, plot_image_heatmap, replace_with_feature
import torch
from PIL import Image
from diffusers import FluxPipeline
from einops import rearrange
from autoencoder import TopkSparseAutoencoder
import re
import numpy as np
import matplotlib.pyplot as plt
import random
# Verify hook signature
import inspect
import traceback
import os
from pathlib import Path


## 1. Configuration and Setup

Load FLUX pipeline, SAE model, and determine hook location based on SAE model selection.


In [ ]:
# Configuration
FLUX_MODEL = "black-forest-labs/FLUX.1-schnell"

# Available HuggingFace SAE models - select by index (0-5)
SAE_MODELS_HF = [
    "RE-N-Y/cc3m-transformer_blocks.0-0",        # 0: MMDiT layer 0, image stream
    "RE-N-Y/cc3m-transformer_blocks.0-1",        # 1: MMDiT layer 0, text stream
    "RE-N-Y/cc3m-transformer_blocks.18-0",       # 2: MMDiT layer 18, image stream
    "RE-N-Y/cc3m-transformer_blocks.18-1",       # 3: MMDiT layer 18, text stream
    "RE-N-Y/cc3m-single_transformer_blocks.9",   # 4: DiT layer 9, fused stream
    "RE-N-Y/cc3m-single_transformer_blocks.37",  # 5: DiT layer 37, fused stream
]

# Local checkpoint folder
# LOCAL_CHECKPOINT_FOLDER = '/mnt/drive_a/Projects/sae/checkpoints/hyperparameter_test'
LOCAL_CHECKPOINT_FOLDER = '/mnt/drive_a/Projects/sae/checkpoints/base_test'

# Discover local checkpoints
local_checkpoints = []
if os.path.exists(LOCAL_CHECKPOINT_FOLDER):
    for item in os.listdir(LOCAL_CHECKPOINT_FOLDER):
        checkpoint_path = os.path.join(LOCAL_CHECKPOINT_FOLDER, item)
        if os.path.isdir(checkpoint_path) and os.path.exists(os.path.join(checkpoint_path, 'model.safetensors')):
            local_checkpoints.append(item)
    local_checkpoints.sort()

print(f"Found {len(local_checkpoints)} local checkpoints:")
for i, cp in enumerate(local_checkpoints):
    print(f"  {i}: {cp}")

# Select model source: 'hf' for HuggingFace, 'local' for local checkpoints
USE_LOCAL_MODEL = True  # Set to False to use HuggingFace models

if USE_LOCAL_MODEL:
    # Select local checkpoint by index
    LOCAL_CHECKPOINT_INDEX = 4 # Change this to select different local checkpoints
    if LOCAL_CHECKPOINT_INDEX < len(local_checkpoints):
        selected_checkpoint = local_checkpoints[LOCAL_CHECKPOINT_INDEX]
        SAE_MODEL = os.path.join(LOCAL_CHECKPOINT_FOLDER, selected_checkpoint)
        print(f"Selected local checkpoint (index {LOCAL_CHECKPOINT_INDEX}): {selected_checkpoint}")
    else:
        print(f"Warning: Local checkpoint index {LOCAL_CHECKPOINT_INDEX} out of range. Using first checkpoint.")
        SAE_MODEL = os.path.join(LOCAL_CHECKPOINT_FOLDER, local_checkpoints[0])
else:
    # Select HuggingFace SAE model by index (0-5)
    SAE_MODEL_INDEX = 5  # Change this index (0-5) to test different SAEs
    SAE_MODEL = SAE_MODELS_HF[SAE_MODEL_INDEX]
    print(f"Selected HuggingFace SAE model (index {SAE_MODEL_INDEX}): {SAE_MODEL}")

prompt = "A cinematic shot of a professor sloth wearing a tuxedo at a BBQ party."
prompt = "a woman with pink hair standing in a forest, holding a sign that says ‘Skipping flux blocks‘"

# prompt = "a woman with pink hair standing in a dark room, holding a sign says `SAE‘. Behind her left is a blue light, right is a round red light. 4K. Photorealistic."
# prompt = 'A faceted silver disco ball (front left) and a white ceramic mug (rear right) sitting on a white desk between two black gooseneck lamps. Right lamp turned on shining bright beam into the mug, glowing interior. Reflections sparkling on the disco ball. Left lamp turned off, dark room, high contrast, photorealistic, 4k.'
# prompt = 'A faceted silver disco ball (front left) and a white ceramic mug (rear right) sitting on a white desk between two black gooseneck lamps. The left lamp is turned on, shining red soft light onto the disco ball, creating red reflections. The right lamp is turned on, shining a bright blue soft light directly into the mug. The white desk surface shows blended red and blue light patterns. Dark room, high contrast, photorealistic, 4k.'
# prompt = 'Faceted silver disco ball front left, white ceramic mug rear right, on white desk between two black gooseneck lamps. Left lamp emitting soft red light onto ball. Right lamp emitting bright blue light into mug, glowing blue interior. Blended red blue light on desk surface. Dark room, high contrast, photorealistic, 4k.'
# prompt = 'The capital of France is'
# prompt = 'The capital of China is'

device = "cuda"
dtype = torch.bfloat16

# Load FLUX pipeline
print("Loading FLUX pipeline...")
pipe = FluxPipeline.from_pretrained(
    FLUX_MODEL,
    torch_dtype=dtype
)
pipe = pipe.to(device)
pipe.set_progress_bar_config(disable=True)  # Disable progress bar

# Load SAE
print(f"Loading SAE: {SAE_MODEL}...")
sae = TopkSparseAutoencoder.from_pretrained(SAE_MODEL)
sae = sae.to(device)
sae.eval()

# Determine hook location and stream from SAE model name
# Pattern 1 (HuggingFace): transformer_blocks.X-Y or single_transformer_blocks.X
# Pattern 2 (Local): transformer_blocks_X_ff_streamY or single_transformer_blocks_X_proj_mlp_streamY
sae_match = re.search(r"(?:transformer_blocks|single_transformer_blocks)[._](\d+)(?:[-_](\d+))?", SAE_MODEL)
if sae_match:
    block_num = sae_match.group(1)
    stream_num = sae_match.group(2)
    
    # Check if it's a single_transformer_blocks (DiT layer)
    if "single_transformer_blocks" in SAE_MODEL:
        hook_location = f"single_transformer_blocks.{block_num}.attn"
        # For local models, check stream from name (e.g., stream0, stream1)
        if stream_num:
            stream = int(stream_num)
        else:
            # Try to extract from local naming pattern: stream0, stream1, etc.
            stream_match = re.search(r"stream(\d+)", SAE_MODEL)
            stream = int(stream_match.group(1)) if stream_match else 0
        is_dit = True
    else:
        hook_location = f"transformer_blocks.{block_num}.attn"
        # For HuggingFace models: X-Y format where Y is stream
        # For local models: extract from streamY pattern
        if stream_num:
            stream = int(stream_num)
        else:
            # Try to extract from local naming pattern: stream0, stream1, etc.
            stream_match = re.search(r"stream(\d+)", SAE_MODEL)
            stream = int(stream_match.group(1)) if stream_match else 0
        is_dit = False
else:
    # Fallback: try to extract from local naming pattern
    stream_match = re.search(r"stream(\d+)", SAE_MODEL)
    if stream_match:
        stream = int(stream_match.group(1))
    else:
        stream = 0
    
    # Try to detect layer type from model name
    if "single_transformer_blocks" in SAE_MODEL:
        block_match = re.search(r"single_transformer_blocks[._](\d+)", SAE_MODEL)
        if block_match:
            hook_location = f"single_transformer_blocks.{block_match.group(1)}.attn"
            is_dit = True
        else:
            hook_location = "single_transformer_blocks.0.attn"
            is_dit = True
    else:
        block_match = re.search(r"transformer_blocks[._](\d+)", SAE_MODEL)
        if block_match:
            hook_location = f"transformer_blocks.{block_match.group(1)}.attn"
            is_dit = False
        else:
            hook_location = "transformer_blocks.0.attn"
            is_dit = False

print(f"🔎 Hook location: {hook_location}")
print(f"🔎 Stream: {stream} (0=query/image, 1=key/text)")
print(f"🔎 Is DiT layer: {is_dit}")


## 2. Image Dimensions and Token Calculations

Calculate spatial dimensions and token counts for the target image resolution. Supports arbitrary M×N dimensions (must be multiples of 16).


In [ ]:
# FLUX Architecture Constants
T_text = 512  # Text tokens are always 512 (fixed by FLUX architecture, independent of image size)

# Image dimensions - must be divisible by 16
# M and N can be different (e.g., 512×768, 1024×512, etc.)
# For random test: choose from common valid sizes

# VALID_SIZES = [512, 768, 1024, 1280, 1536]
VALID_SIZES = [128, 256, 512]
M = random.choice(VALID_SIZES)  # Height
N = random.choice(VALID_SIZES)  # Width (can be different from M)

M = 512
N = 512

# Validate dimensions
assert M % 16 == 0, f"M ({M}) must be divisible by 16"
assert N % 16 == 0, f"N ({N}) must be divisible by 16"

# Calculate image token dimensions
# FLUX reduces by factor of 16: h = M/16, w = N/16
h_img = M // 16  # Spatial height in tokens
w_img = N // 16  # Spatial width in tokens
T_img = h_img * w_img  # Total image tokens = (M × N) / 256

# For DiT fused stream: [T_text text tokens, T_img image tokens]
T_fused = T_text + T_img  # Total tokens in fused stream

print(f"📐 Image dimensions: {M}×{N} pixels (M≠N allowed)")
print(f"📐 Token grid: {h_img}×{w_img} = {T_img} image tokens")
print(f"📐 Text tokens: {T_text} (fixed)")
print(f"📐 Fused stream (DiT): {T_fused} tokens ({T_text} text + {T_img} image)")

# # Random prompt for testing
# PROMPTS = [
#     "A cinematic shot of a professor sloth wearing a tuxedo at a BBQ party.",
#     "a woman with pink hair standing in a forest, holding a sign that says 'Skipping flux blocks'",
#     "A futuristic cityscape at sunset with flying cars and neon lights",
#     "A serene mountain landscape with a crystal clear lake reflecting the peaks",
#     "An abstract painting with vibrant colors and geometric shapes",
#     "A cozy coffee shop interior with warm lighting and bookshelves",
#     "A space station orbiting a distant planet with stars in the background",
#     "A vintage car driving through a desert road at golden hour",
# ]

# PROMPTS = [
#     "A square color palette (in RGB color space), with pricsely segmented in to four sections, each section has a different color, the colors are red([0,0,255] top left), green([0,255,0]top right), blue([255,0,0]bottom left) and yellow([0,255, 255]bottom right)"
# ]
# PROMPTS = [
#     "A square color palette divided into four equal sections: red top-left, green top-right, blue bottom-left, yellow bottom-right"
# ]

# prompt = random.choice(PROMPTS)
print(f"📝 Random prompt: {prompt}")

# Random seed for reproducibility (but different each run)
# seed = random.randint(0, 2**32 - 1)
seed = 42
print(f"🎲 Random seed: {seed}")


## 3. Generate Base Image

Generate an image using the prompt to establish a baseline for comparison.


In [ ]:
# Generate an image by prompt
generator = torch.Generator(device=device).manual_seed(seed)
test_timesteps = 4
print(f"Generating {M}×{N} image...")
output = pipe(
    prompt=prompt,
    height=M,
    width=N,
    num_inference_steps=test_timesteps,
    guidance_scale=0.0,
    generator=generator,
)

image = output.images[0]
image


## 4. Generate Sparse Maps

Capture activations from the transformer layer and encode them using the SAE to generate sparse activation maps.

**Configuration**: Set `USE_RESIDUAL` to switch between:
- `False` (default): Direct activations (output only)
- `True`: Residual-based sparse maps (output - input), similar to SDXL method

The residual method captures what the layer *adds* rather than absolute values, which can be more informative and aligned with SDXL training approaches.


In [ ]:
# Generate sparse maps by hooking into the transformer
# Configuration: Choose between direct activations or residual-based (SDXL-style)
USE_RESIDUAL = False  # Set to True to use residual (output - input), False for direct activations
activation_hook_input = None  # For residual computation
activation_hook_output = None

def previous_layer_hook(module, input, output):
    """Hook to capture the previous layer's output (which becomes input to current layer)
    
    This is a forward hook, so it receives (module, input, output).
    """
    global activation_hook_input
    
    if USE_RESIDUAL:
        # Capture output from previous layer
        if isinstance(output, tuple):
            query, key = output
            # Select the appropriate stream
            activation_hook_input = query if stream == 0 else key
        else:
            activation_hook_input = output
    # Forward hooks should return output (or None to not modify)
    return output

def activation_hook(module, input, output):
    """Hook to capture output after the module processes it
    
    This is a forward hook, so it receives (module, input, output).
    """
    global activation_hook_output
    
    # Capture output
    if isinstance(output, tuple):
        query, key = output
        # Select the appropriate stream
        activation_hook_output = query if stream == 0 else key
    else:
        activation_hook_output = output
    # Forward hooks should return output (or None to not modify)
    return output

# Create a wrapper to ensure correct signature and prevent accidental pre-hook registration
def activation_hook_wrapper(*args):
    """Wrapper to ensure activation_hook is called with correct signature
    
    This wrapper detects if it's being called as a pre-hook (2 args) or forward hook (3 args).
    """
    if len(args) == 2:
        # This was called as a pre-hook (2 args), which is wrong
        raise TypeError(
            f"activation_hook_wrapper was called as a pre-hook with 2 arguments, "
            f"but it must be registered as a forward hook (3 arguments). "
            f"Received {len(args)} arguments: {[type(a) for a in args]}. "
            f"This hook must be registered with register_forward_hook, not register_forward_pre_hook!"
        )
    elif len(args) == 3:
        # Normal forward hook call (3 args: module, input, output)
        module, input, output = args
        return activation_hook(module, input, output)
    else:
        raise TypeError(f"activation_hook_wrapper received unexpected number of arguments: {len(args)}")

# Register hooks
# Philosophy: For residual computation, we measure the previous layer's output as input
# This is more reliable than parsing input tuples, especially for attention modules
previous_layer_hook_handle = None
hook_handle = None

# Initialize hook variables
activation_hook_input = None
activation_hook_output = None

if USE_RESIDUAL:
    # Calculate previous layer index
    current_block_idx = int(block_num) if isinstance(block_num, str) else block_num
    
    if current_block_idx > 0:
        # Hook into the previous layer to capture its output
        prev_block_idx = current_block_idx - 1
        
        if "single_transformer_blocks" in SAE_MODEL:
            # For DiT layers: single_transformer_blocks
            previous_layer_location = f"single_transformer_blocks.{prev_block_idx}.attn"
        else:
            # For MMDiT layers: transformer_blocks
            # Need to handle stream selection for MMDiT
            previous_layer_location = f"transformer_blocks.{prev_block_idx}.attn"
        
        try:
            prev_module = pipe.transformer.get_submodule(previous_layer_location)
            
            # Clear any existing hooks on previous module (safety measure)
            if hasattr(prev_module, '_forward_hooks'):
                prev_module._forward_hooks.clear()
            if hasattr(prev_module, '_forward_pre_hooks'):
                prev_module._forward_pre_hooks.clear()
            
            sig = inspect.signature(previous_layer_hook)
            expected_params = ['module', 'input', 'output']
            actual_params = list(sig.parameters.keys())
            if actual_params != expected_params:
                raise ValueError(f"Previous layer hook signature mismatch! Expected {expected_params}, got {actual_params}")
            
            previous_layer_hook_handle = prev_module.register_forward_hook(previous_layer_hook)
            
            # Verify it was registered as a forward hook
            if hasattr(prev_module, '_forward_hooks') and len(prev_module._forward_hooks) > 0:
                print(f"✅ Registered previous layer hook: {previous_layer_location} (forward hook)")
            else:
                raise RuntimeError("Previous layer hook was not registered as a forward hook!")
                
        except Exception as e:
            print(f"⚠️  Could not register previous layer hook: {e}")
            print("   Residual mode will use direct activations (output only)")
            previous_layer_hook_handle = None
    else:
        print("⚠️  Current layer is the first layer (index 0), no previous layer to hook")
        print("   Residual mode will use direct activations (output only)")

# Register hook for current layer output
# Make sure we're registering a forward hook (not pre-hook) with correct signature
try:
    current_module = pipe.transformer.get_submodule(hook_location)
    
    # Clear any existing hooks on this module first (safety measure)
    if hasattr(current_module, '_forward_hooks'):
        current_module._forward_hooks.clear()
    if hasattr(current_module, '_forward_pre_hooks'):
        current_module._forward_pre_hooks.clear()
    
    # Verify hook signature is correct for forward hook: (module, input, output)
    sig = inspect.signature(activation_hook)
    expected_params = ['module', 'input', 'output']
    actual_params = list(sig.parameters.keys())
    if actual_params != expected_params:
        raise ValueError(f"Hook signature mismatch! Expected {expected_params}, got {actual_params}")
    
    # Explicitly register as forward hook (not pre-hook)
    # Use wrapper to ensure correct signature
    hook_handle = current_module.register_forward_hook(activation_hook_wrapper)
    
    # Verify it was registered as a forward hook, not pre-hook
    if hasattr(current_module, '_forward_hooks') and len(current_module._forward_hooks) > 0:
        print(f"✅ Registered current layer hook: {hook_location} (forward hook)")
    else:
        raise RuntimeError("Hook was not registered as a forward hook!")
        
except Exception as e:
    print(f"❌ Error registering current layer hook: {e}")
    traceback.print_exc()
    raise

# Generate once to capture activations
print("Capturing activations...")
_ = pipe(
    prompt=prompt,
    height=M,
    width=N,
    num_inference_steps=test_timesteps,  # Just 1 step to get activations
    guidance_scale=0.0,
    generator=generator,
)

# Remove hooks
hook_handle.remove()
if previous_layer_hook_handle is not None:
    previous_layer_hook_handle.remove()

def generate_sparse_maps(activation_output, sae, is_dit=False, stream=0, h_img=None, w_img=None, T_img=None, 
                         activation_input=None, use_residual=False):
    """
    Generate sparse maps from activations.
    
    Args:
        activation_output: Captured activations from hook (module output)
        sae: The SAE model
        is_dit: Whether this is a DiT layer (fused stream)
        stream: Stream number (0=image/query, 1=text/key)
        h_img: Spatial height in tokens (M/16)
        w_img: Spatial width in tokens (N/16)
        T_img: Total image tokens (h_img * w_img)
        activation_input: Captured activations from hook (module input) for residual computation
        use_residual: If True, compute residual (output - input) before encoding
    
    Returns:
        sparse_maps: Sparse activation maps (spatial for image, flat for text)
        spatial_shape: (h, w) for image streams, None for text streams
        sparse_maps_flat: Full flat sparse maps (for DiT steering) or None
    """
    if activation_output is None:
        raise ValueError("No activations captured. Make sure the hook was registered.")
    
    # Compute residual if enabled
    if use_residual and activation_input is not None:
        # Check if shapes match for residual computation
        try:
            # Ensure same shape and dtype
            if activation_input.shape != activation_output.shape:
                print(f"⚠️  Input shape {activation_input.shape} != output shape {activation_output.shape}")
                print("   Falling back to direct activations (output only)")
                activations_to_encode = activation_output
            else:
                # Compute residual: output - input
                activations_to_encode = activation_output - activation_input
                print("✅ Using residual-based sparse maps (output - input)")
        except Exception as e:
            print(f"⚠️  Error computing residual: {e}")
            print("   Falling back to direct activations (output only)")
            activations_to_encode = activation_output
    else:
        activations_to_encode = activation_output
        if use_residual:
            print("⚠️  Residual mode enabled but no input captured. Using direct activations.")
    
    # Flatten activations: (batch, seq, h, w, dim) or (batch, seq, length, dim) -> (batch*seq*h*w, dim)
    original_shape = activations_to_encode.shape
    activations_flat = rearrange(activations_to_encode, "b ... d -> (b ...) d")
    
    # Ensure dtype/device match
    sae_dtype = next(sae.parameters()).dtype
    sae_device = next(sae.parameters()).device
    activations_flat = activations_flat.to(dtype=sae_dtype, device=sae_device)
    
    # Encode with SAE
    with torch.no_grad():
        sparse_maps_flat = sae.encode(activations_flat)  # Shape: (num_tokens, num_features)
    
    # Handle DiT layers (fused stream: text + image concatenated)
    if is_dit:
        # DiT: sequence is [text_tokens (512), image_tokens (T_img)]
        num_text_tokens = 512
        if T_img is None:
            # Fallback: try to infer from total tokens
            total_tokens = sparse_maps_flat.shape[0]
            T_img = total_tokens - num_text_tokens
        
        text_sparse_maps = sparse_maps_flat[:num_text_tokens, :]
        image_sparse_maps = sparse_maps_flat[num_text_tokens:num_text_tokens + T_img, :]
        
        # Reshape image tokens to spatial dimensions
        if h_img is None or w_img is None:
            # Try to infer from T_img
            h_img = int(np.sqrt(T_img))
            w_img = T_img // h_img
            if h_img * w_img != T_img:
                # Fallback: find reasonable dimensions
                for h in range(8, int(np.sqrt(T_img)) + 1):
                    w = T_img // h
                    if h * w == T_img:
                        h_img, w_img = h, w
                        break
        
        image_sparse_maps = image_sparse_maps.reshape(h_img, w_img, -1)
        
        # Return image sparse maps for visualization, and full flat maps for steering
        return image_sparse_maps, (h_img, w_img), sparse_maps_flat
    else:
        # MMDiT layers
        if stream == 0:
            # Image stream: reshape to spatial dimensions
            num_tokens = sparse_maps_flat.shape[0]
            if h_img is not None and w_img is not None and num_tokens == h_img * w_img:
                sparse_maps = sparse_maps_flat.reshape(h_img, w_img, -1)
                return sparse_maps, (h_img, w_img), None
            else:
                # Fallback: try to infer spatial dimensions
                h = int(np.sqrt(num_tokens))
                w = num_tokens // h
                if h * w == num_tokens:
                    sparse_maps = sparse_maps_flat.reshape(h, w, -1)
                    return sparse_maps, (h, w), None
                else:
                    # Can't reshape to 2D, return as 1D
                    return sparse_maps_flat, None, None
        else:
            # Text stream: keep as sequence (512, features)
            # Note: padding tokens should be filtered when analyzing
            return sparse_maps_flat, None, None

# Generate sparse maps
sparse_maps, spatial_shape, sparse_maps_flat = generate_sparse_maps(
    activation_hook_output, sae, is_dit=is_dit, stream=stream, 
    h_img=h_img, w_img=w_img, T_img=T_img,
    activation_input=activation_hook_input, use_residual=USE_RESIDUAL
)
print(f"Sparse maps shape: {sparse_maps.shape}")
if spatial_shape:
    print(f"Spatial shape: {spatial_shape}")
if sparse_maps_flat is not None:
    print(f"Full sparse maps shape (for DiT): {sparse_maps_flat.shape}")
    if is_dit:
        print(f"  - Text tokens: 0-511 (512 tokens)")
        print(f"  - Image tokens: 512-{512+T_img-1} ({T_img} tokens)")


## 5. Compute Mean Activations

Compute mean activation strength per feature for use in empty-prompt intervention and other analyses.


In [ ]:
# Compute mean activations per feature (for empty-prompt intervention)
# This is needed for empty-prompt intervention similar to SDXL
print("Computing mean activations per feature...")
if len(sparse_maps.shape) == 3:
    # Spatial: mean across spatial dimensions
    feature_means = sparse_maps.mean(axis=(0, 1))
elif len(sparse_maps.shape) == 2:
    # Sequence: mean across sequence dimension
    feature_means = sparse_maps.mean(axis=0)
else:
    feature_means = sparse_maps.mean(axis=0)

# Store mean activations (similar to SDXL's means_dict)
means = feature_means.cpu()  # Store on CPU to save GPU memory
print(f"Mean activations shape: {means.shape}")
print(f"Mean activation range: [{means.min():.4f}, {means.max():.4f}]")


## 6. DiT Text Token Analysis (DiT Layers Only)

For DiT (fused stream) layers, analyze text token activations and extract self-attention maps to convert text features to image space.


In [ ]:
# DiT Text Token Analysis (only for DiT modules)
# This cell analyzes text tokens in DiT layers and converts them to image space via cross-attention
if is_dit and sparse_maps_flat is not None:
    print("="*80)
    print("📝 DiT Text Token Analysis - Extracting Text and Image Tokens")
    print("="*80)
    
    # Extract text and image tokens from DiT fused stream
    text_sparse_maps = sparse_maps_flat[:T_text, :]  # First 512 tokens (text)
    image_sparse_maps_flat = sparse_maps_flat[T_text:T_text + T_img, :]  # Next T_img tokens (image)
    
    print(f"✅ Extracted text sparse maps: {text_sparse_maps.shape}")
    print(f"✅ Extracted image sparse maps (flat): {image_sparse_maps_flat.shape}")
    
    # Get tokenizer to decode token texts (for visualization)
    can_decode_tokens = False
    tokenizer = None
    dit_token_texts = {}
    dit_token_ids_list = []
    
    try:
        # Try to get tokenizer from FLUX pipeline
        if hasattr(pipe, 'tokenizer'):
            tokenizer = pipe.tokenizer
            can_decode_tokens = True
        elif hasattr(pipe, 'text_encoder') and hasattr(pipe.text_encoder, 'tokenizer'):
            tokenizer = pipe.text_encoder.tokenizer
            can_decode_tokens = True
        else:
            # Fallback: try to load T5 tokenizer
            from transformers import T5Tokenizer
            tokenizer = T5Tokenizer.from_pretrained("google/t5-v1_1-xxl")
            can_decode_tokens = True
    except Exception as e:
        can_decode_tokens = False
        print(f"⚠️  Tokenizer not available - showing token indices only ({str(e)})")
    
    # Get actual prompt tokens (exclude padding)
    if can_decode_tokens:
        try:
            # Tokenize the prompt to get actual token IDs
            tokenized = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=T_text)
            dit_token_ids_list = tokenized['input_ids'][0].cpu().tolist()
            dit_actual_prompt_length = len([tid for tid in dit_token_ids_list if tid != tokenizer.pad_token_id])
            
            # Decode tokens to text
            for i, token_id in enumerate(dit_token_ids_list[:dit_actual_prompt_length]):
                try:
                    token_text = tokenizer.decode([token_id])
                    dit_token_texts[i] = token_text.strip()
                except:
                    dit_token_texts[i] = f"<token_{i}>"
        except Exception as e:
            print(f"⚠️  Could not tokenize prompt: {str(e)}")
            dit_actual_prompt_length = T_text  # Fallback: assume all tokens are used
    else:
        dit_actual_prompt_length = T_text  # Fallback: assume all tokens are used
    
    print(f"📝 Actual DiT text prompt length: {dit_actual_prompt_length} tokens (out of {T_text} total slots)")
    if dit_actual_prompt_length < T_text:
        print(f"⚠️  Indices {dit_actual_prompt_length}-{T_text-1} are padding tokens - will be filtered from analysis")
    
    # 1. Analyze Text Token Activations
    print("\n1️⃣  Analyzing Text Token Top Activations...")
    
    # Filter out padding tokens for analysis
    text_sparse_maps_for_analysis = text_sparse_maps[:dit_actual_prompt_length, :]
    
    # Compute top features from text tokens only
    if text_sparse_maps_for_analysis.dtype == torch.bfloat16:
        text_sparse_maps_for_analysis = text_sparse_maps_for_analysis.float()
    
    # Get top features from text tokens
    text_feature_means = text_sparse_maps_for_analysis.mean(axis=0)
    num_top_features = 10
    text_top_features = text_feature_means.topk(num_top_features).indices.cpu().tolist()
    
    print(f"✅ Top {num_top_features} features from text tokens: {text_top_features}")
    
    # Plot: Token Position vs Feature Activation
    # X-axis: 512 T5 tokens (but focus on first few prompt tokens)
    # Y-axis: Top features
    # Color: Activation strength
    
    # Focus on first few prompt tokens (e.g., first 20-30 tokens)
    max_tokens_to_plot = min(30, dit_actual_prompt_length)
    tokens_to_plot = list(range(max_tokens_to_plot))
    
    # Create activation matrix: (tokens, features)
    activation_matrix = text_sparse_maps_for_analysis[:max_tokens_to_plot, text_top_features].cpu().numpy()
    
    # Plot heatmap
    plt.figure(figsize=(14, 8))
    plt.imshow(activation_matrix.T, aspect='auto', cmap='hot', interpolation='nearest')
    plt.colorbar(label='Activation Strength')
    plt.xlabel('Token Position (T5 tokens)', fontsize=12, fontweight='bold')
    plt.ylabel('Top Features', fontsize=12, fontweight='bold')
    plt.title(f'DiT Text Tokens: Top {num_top_features} Features - First {max_tokens_to_plot} Tokens in Prompt Order', 
              fontsize=14, fontweight='bold')
    
    # Set x-axis labels to show token texts
    if can_decode_tokens and len(dit_token_texts) > 0:
        tick_labels = []
        for i in tokens_to_plot:
            token_text = dit_token_texts.get(i, f"T{i}")
            if len(token_text) > 10:
                token_text = token_text[:10] + "..."
            tick_labels.append(f"{i}\n{token_text}")
        plt.xticks(range(max_tokens_to_plot), tick_labels, rotation=45, ha='right', fontsize=8)
    else:
        plt.xticks(range(max_tokens_to_plot), [f"T{i}" for i in tokens_to_plot], rotation=45, ha='right', fontsize=8)
    
    # Set y-axis labels to show feature indices
    plt.yticks(range(num_top_features), [f"F{text_top_features[i]}" for i in range(num_top_features)], fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Print token list
    print(f"\n📝 DiT Text Prompt Tokens (first {max_tokens_to_plot} tokens):")
    print("="*80)
    for i in tokens_to_plot:
        token_text = dit_token_texts.get(i, f"<token_{i}>")
        activation_avg = float(text_sparse_maps_for_analysis[i, :].mean().item())
        print(f"  Token {i:3d}: {token_text:30s} | Avg Activation: {activation_avg:.4f}")
    if dit_actual_prompt_length > max_tokens_to_plot:
        print(f"  ... (showing first {max_tokens_to_plot} of {dit_actual_prompt_length} actual tokens)")
    print("="*80)
    
    # 2. Token Index vs Activation Strength Plot
    print("\n2️⃣  Generating Token Index vs Activation Strength Plot...")
    
    # Compute average activation strength per token (across all features)
    token_activation_strengths = text_sparse_maps_for_analysis.mean(axis=1).cpu().numpy()  # (num_tokens,)
    
    # Create line plot: Token Index vs Activation Strength
    plt.figure(figsize=(14, 6))
    token_indices_plot = np.arange(len(token_activation_strengths))
    plt.plot(token_indices_plot, token_activation_strengths, marker='o', markersize=4, linewidth=1.5)
    plt.xlabel('Token Index (T5 tokens)', fontsize=12, fontweight='bold')
    plt.ylabel('Average Activation Strength', fontsize=12, fontweight='bold')
    plt.title(f'Token Index vs Activation Strength - First {len(token_activation_strengths)} Tokens', 
              fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    # Add token text labels for first few tokens
    if can_decode_tokens and len(dit_token_texts) > 0:
        max_labels = min(20, len(token_activation_strengths))
        for i in range(max_labels):
            token_text = dit_token_texts.get(i, f"T{i}")
            if len(token_text) > 15:
                token_text = token_text[:15] + "..."
            plt.annotate(token_text, 
                        (i, token_activation_strengths[i]),
                        textcoords="offset points", 
                        xytext=(0,10), 
                        ha='center',
                        fontsize=7,
                        rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Also create a bar plot for better visibility
    plt.figure(figsize=(14, 6))
    plt.bar(token_indices_plot, token_activation_strengths, alpha=0.7, width=0.8)
    plt.xlabel('Token Index (T5 tokens)', fontsize=12, fontweight='bold')
    plt.ylabel('Average Activation Strength', fontsize=12, fontweight='bold')
    plt.title(f'Token Index vs Activation Strength (Bar Plot) - First {len(token_activation_strengths)} Tokens', 
              fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add token text labels on x-axis for first few tokens
    if can_decode_tokens and len(dit_token_texts) > 0:
        max_labels = min(30, len(token_activation_strengths))
        tick_positions = list(range(max_labels))
        tick_labels = []
        for i in range(max_labels):
            token_text = dit_token_texts.get(i, f"T{i}")
            if len(token_text) > 10:
                token_text = token_text[:10] + "..."
            tick_labels.append(f"{i}\n{token_text}")
        plt.xticks(tick_positions, tick_labels, rotation=45, ha='right', fontsize=8)
    else:
        plt.xticks(token_indices_plot[::max(1, len(token_indices_plot)//30)], 
                  [f"T{i}" for i in token_indices_plot[::max(1, len(token_indices_plot)//30)]],
                  rotation=45, ha='right', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    print(f"✅ Token activation strength plot generated")
    print(f"   Max activation: {token_activation_strengths.max():.4f} at token {token_activation_strengths.argmax()}")
    print(f"   Min activation: {token_activation_strengths.min():.4f} at token {token_activation_strengths.argmin()}")
    print(f"   Mean activation: {token_activation_strengths.mean():.4f}")
    
    # 3. Extract Self-Attention Matrix from DiT Layer to Convert Text Tokens to Image Space
    print("\n3️⃣  Extracting Self-Attention Matrix from DiT Layer for Text-to-Image Conversion...")
    print("   💡 DiT uses global self-attention on fused sequence [Text, Image]")
    print("   💡 We extract the bottom-left quadrant: Image Queries × Text Keys")
    print("   💡 This shows how image positions attend to text tokens")
    
    def extract_dit_self_attention_maps(prompt, num_steps=1, max_tokens=None):
        """
        Extract self-attention maps from DiT layer for text tokens.
        DiT uses global self-attention on fused sequence [Text (512), Image (T_img)].
        We extract the bottom-left quadrant: Image Queries × Text Keys.
        Returns attention maps: (height, width) for each text token.
        """
        attention_maps = {}
        
        # Determine which tokens to extract (focus on first few prompt tokens)
        if max_tokens is not None:
            token_indices = list(range(min(max_tokens, dit_actual_prompt_length)))
        else:
            token_indices = list(range(dit_actual_prompt_length))
        
        print(f"   Extracting attention maps for {len(token_indices)} text tokens")
        print(f"   Attention matrix shape: ({T_text + T_img}, {T_text + T_img})")
        print(f"   Extracting quadrant: Image Queries [{T_text}:{T_text + T_img}] × Text Keys [0:{T_text}]")
        
        def attention_hook(module, input, output):
            """Hook to extract self-attention weights from DiT"""
            # For DiT, we need to access the self-attention matrix
            # The attention matrix has shape (batch, num_heads, num_queries, num_keys)
            # where num_queries = num_keys = T_text + T_img (fused sequence)
            
            # Method 1: Try to access stored attention weights
            if hasattr(module, 'last_attention_weights'):
                attn_weights = module.last_attention_weights
                if attn_weights is not None:
                    # attn_weights shape: (batch, num_heads, T_text+T_img, T_text+T_img)
                    attn_weights = attn_weights.mean(dim=1)  # Average over heads: (batch, T_text+T_img, T_text+T_img)
                    
                    # Extract bottom-left quadrant: Image Queries × Text Keys
                    # Rows (Queries): Image tokens [T_text : T_text+T_img]
                    # Columns (Keys): Text tokens [0 : T_text]
                    image_text_attention = attn_weights[0, T_text:T_text + T_img, :T_text]  # (T_img, T_text)
                    
                    # For each text token, get attention from all image positions
                    for token_idx in token_indices:
                        if token_idx < T_text:
                            # Get attention from all image positions to this text token
                            token_attention = image_text_attention[:, token_idx]  # (T_img,)
                            
                            if token_attention.dtype == torch.bfloat16:
                                token_attention = token_attention.float()
                            
                            # Reshape to spatial dimensions
                            try:
                                attention_map = token_attention.reshape(h_img, w_img).cpu().numpy()
                                attention_maps[token_idx] = attention_map
                            except Exception as e:
                                print(f"   ⚠️  Error reshaping token {token_idx}: {e}")
            
            # Method 2: Compute attention from input (if we can access query/key)
            # This is a fallback if attention weights aren't stored
            elif isinstance(input, tuple) and len(input) >= 1:
                # Try to compute attention from the input hidden states
                # For DiT, input[0] should be the hidden states of shape (batch, T_text+T_img, dim)
                hidden_states = input[0] if len(input) > 0 else None
                
                if hidden_states is not None and len(hidden_states.shape) == 3:
                    batch, seq_len, dim = hidden_states.shape
                    
                    # Check if this matches our expected fused sequence length
                    if seq_len == T_text + T_img:
                        # We need to access the attention module's query/key projections
                        # This is a simplified approach - we compute QK^T from hidden states
                        # Note: This assumes the attention module has q_proj and k_proj
                        if hasattr(module, 'q_proj') and hasattr(module, 'k_proj'):
                            hidden_comp = hidden_states.float() if hidden_states.dtype == torch.bfloat16 else hidden_states
                            
                            # Project to query and key
                            queries = module.q_proj(hidden_comp)  # (batch, seq_len, dim)
                            keys = module.k_proj(hidden_comp)     # (batch, seq_len, dim)
                            
                            # Compute attention scores: Q @ K^T / sqrt(dim)
                            scale = 1.0 / np.sqrt(float(dim))
                            attention_scores = torch.matmul(queries, keys.transpose(-2, -1)) * scale
                            attention_scores = torch.softmax(attention_scores, dim=-1)  # (batch, seq_len, seq_len)
                            
                            # Extract bottom-left quadrant: Image Queries × Text Keys
                            image_text_attention = attention_scores[0, T_text:T_text + T_img, :T_text]  # (T_img, T_text)
                            
                            # For each text token, get attention from all image positions
                            for token_idx in token_indices:
                                if token_idx < T_text:
                                    token_attention = image_text_attention[:, token_idx]  # (T_img,)
                                    
                                    if token_attention.dtype == torch.bfloat16:
                                        token_attention = token_attention.float()
                                    
                                    try:
                                        attention_map = token_attention.reshape(h_img, w_img).cpu().numpy()
                                        attention_maps[token_idx] = attention_map
                                    except Exception as e:
                                        print(f"   ⚠️  Error reshaping token {token_idx}: {e}")
        
        # Register hook on the DiT attention module (the same one we're analyzing)
        # Use the hook_location that was determined earlier (e.g., single_transformer_blocks.37.attn)
        try:
            dit_attn_module = pipe.transformer.get_submodule(hook_location)
            hook_handle = dit_attn_module.register_forward_hook(attention_hook)
            print(f"   ✅ Successfully hooked into {hook_location}")
        except Exception as e:
            print(f"   ⚠️  Could not hook into {hook_location}: {e}")
            return {}, []
        
        # Generate to capture attention
        try:
            _ = pipe(
                prompt=prompt,
                height=M,
                width=N,
                num_inference_steps=num_steps,
                guidance_scale=0.0,
                generator=generator,
            )
        finally:
            # Remove hook if it was successfully registered
            if 'hook_handle' in locals():
                hook_handle.remove()
        
        return attention_maps, token_indices
    
    # Extract self-attention maps from DiT layer
    max_tokens_for_attention = min(20, dit_actual_prompt_length)  # Focus on first 20 tokens
    try:
        dit_attention_maps_dict, dit_token_indices = extract_dit_self_attention_maps(
            prompt, num_steps=1, max_tokens=max_tokens_for_attention
        )
        
        if dit_attention_maps_dict:
            print(f"   ✅ Extracted attention maps for {len(dit_attention_maps_dict)} text tokens")
            
            # 3. Convert Text Token Activations to Image Space Sparse Maps
            print("\n3️⃣  Converting Text Token Activations to Image Space...")
            
            # For each top feature, aggregate text token activations weighted by attention maps
            text_guided_sparse_maps = torch.zeros(h_img, w_img, sparse_maps.shape[2], 
                                                   device=sparse_maps.device, dtype=sparse_maps.dtype)
            
            # Aggregate: for each image position, sum over text tokens weighted by attention
            for token_idx in dit_token_indices:
                if token_idx in dit_attention_maps_dict:
                    attention_map = torch.from_numpy(dit_attention_maps_dict[token_idx]).to(
                        device=sparse_maps.device, dtype=sparse_maps.dtype
                    )
                    # Get text token activations for this token
                    token_activations = text_sparse_maps[token_idx, :]  # (features,)
                    
                    # Weight by attention: attention_map (h, w) * token_activations (features)
                    # Result: (h, w, features)
                    for feat_idx in range(sparse_maps.shape[2]):
                        text_guided_sparse_maps[:, :, feat_idx] += attention_map * token_activations[feat_idx]
            
            print(f"   ✅ Created text-guided sparse maps: {text_guided_sparse_maps.shape}")
            
            # Store for later use in visualization
            text_guided_sparse_maps_available = True
            
            # Option to choose visualization mode
            print("\n💡 Visualization Options:")
            print("   - Use 'image_guided' for original image token sparse maps")
            print("   - Use 'text_guided' for text token activations converted to image space")
            visualization_mode = "image_guided"  # Change to "text_guided" to use text-based heatmaps
            
        else:
            print("   ⚠️  Could not extract attention maps - using image-guided only")
            text_guided_sparse_maps_available = False
            visualization_mode = "image_guided"
            
    except Exception as e:
        print(f"   ⚠️  Error extracting self-attention maps: {str(e)}")
        traceback.print_exc()
        text_guided_sparse_maps_available = False
        visualization_mode = "image_guided"
    
    print("\n✅ DiT Text Token Analysis Complete")
    print("="*80)
    
else:
    print("ℹ️  Skipping DiT Text Token Analysis (not a DiT layer)")
    text_guided_sparse_maps_available = False
    visualization_mode = "image_guided"
    # Initialize variables to avoid NameError in later cells
    text_guided_sparse_maps = None
    dit_attention_maps_dict = {}
    dit_token_indices = []


## 7. MMDiT Text Stream Cross-Attention Analysis (MMDiT Text Stream Only)

For MMDiT text stream SAEs, extract cross-attention maps to convert text stream features to image space using the Feature → Top Token → Cross-Attention → Heatmap strategy.


In [ ]:
# MMDiT Text Stream Cross-Attention Analysis (for text stream SAEs)
# This converts text stream features to image space using cross-attention
if not is_dit and stream == 1 and len(sparse_maps.shape) == 2:
    print("="*80)
    print("📝 MMDiT Text Stream Analysis - Converting Features to Image Space via Cross-Attention")
    print("="*80)
    
    # sparse_maps is 2D: (num_tokens, num_features) for text stream
    text_sparse_maps = sparse_maps  # (512, num_features)
    
    print(f"✅ Text sparse maps shape: {text_sparse_maps.shape}")
    print(f"💡 Strategy: Feature → Top Token → Cross-Attention → Image Heatmap")
    
    # Get actual prompt length (exclude padding tokens)
    can_decode_tokens = False
    tokenizer = None
    mmdit_token_texts = {}
    mmdit_actual_prompt_length = T_text  # Default to 512
    
    try:
        if hasattr(pipe, 'tokenizer'):
            tokenizer = pipe.tokenizer
            can_decode_tokens = True
        elif hasattr(pipe, 'text_encoder') and hasattr(pipe.text_encoder, 'tokenizer'):
            tokenizer = pipe.text_encoder.tokenizer
            can_decode_tokens = True
        else:
            from transformers import T5Tokenizer
            tokenizer = T5Tokenizer.from_pretrained("google/t5-v1_1-xxl")
            can_decode_tokens = True
    except Exception as e:
        can_decode_tokens = False
        print(f"⚠️  Tokenizer not available: {str(e)}")
    
    if can_decode_tokens:
        try:
            tokenized = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=T_text)
            token_ids_list = tokenized['input_ids'][0].cpu().tolist()
            padding_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
            mmdit_actual_prompt_length = len([tid for tid in token_ids_list if tid != padding_token_id])
            
            for i, token_id in enumerate(token_ids_list[:mmdit_actual_prompt_length]):
                try:
                    token_text = tokenizer.decode([token_id])
                    mmdit_token_texts[i] = token_text.strip()
                except:
                    mmdit_token_texts[i] = f"<token_{i}>"
        except Exception as e:
            print(f"⚠️  Could not tokenize prompt: {str(e)}")
    
    print(f"📝 Actual text prompt length: {mmdit_actual_prompt_length} tokens (out of {T_text} total)")
    
    # Extract cross-attention maps from MMDiT layer
    print("\n🔍 Extracting Cross-Attention Maps from MMDiT Layer...")
    print(f"   Hook location: {hook_location}")
    print(f"   Image Queries attend to Text Keys")
    
    def extract_mmdit_cross_attention_maps(prompt, num_steps=1):
        """
        Extract cross-attention maps from MMDiT attention layer.
        For MMDiT: output is (image_stream, text_stream)
        Image stream (queries) attends to text stream (keys).
        Returns attention maps: (height, width) for each text token.
        """
        attention_maps = {}
        
        def attention_hook(module, input, output):
            """Hook to extract cross-attention weights from MMDiT"""
            # For MMDiT, output is a tuple (image_stream, text_stream)
            if isinstance(output, tuple) and len(output) == 2:
                image_stream, text_stream = output
                
                if len(image_stream.shape) == 3 and len(text_stream.shape) == 3:
                    batch, num_queries, dim = image_stream.shape  # Image tokens (queries)
                    batch_k, num_keys, dim_k = text_stream.shape  # Text tokens (keys)
                    
                    # Ensure compatible dtypes
                    image_stream_comp = image_stream.float() if image_stream.dtype == torch.bfloat16 else image_stream
                    text_stream_comp = text_stream.float() if text_stream.dtype == torch.bfloat16 else text_stream
                    
                    # Compute attention scores: Q @ K^T / sqrt(dim)
                    # Image queries attend to text keys
                    scale = 1.0 / np.sqrt(float(dim))
                    attention_scores = torch.matmul(image_stream_comp, text_stream_comp.transpose(-2, -1)) * scale
                    attention_scores = torch.softmax(attention_scores, dim=-1)  # (batch, num_queries, num_keys)
                    
                    # Store attention for all text tokens
                    for token_idx in range(min(num_keys, mmdit_actual_prompt_length)):
                        # Get attention from all image positions to this text token
                        token_attention = attention_scores[0, :, token_idx]  # (num_queries,)
                        
                        if token_attention.dtype == torch.bfloat16:
                            token_attention = token_attention.float()
                        
                        # Reshape to spatial dimensions
                        try:
                            attention_map = token_attention.reshape(h_img, w_img).cpu().numpy()
                            attention_maps[token_idx] = attention_map
                        except Exception as e:
                            print(f"   ⚠️  Error reshaping token {token_idx}: {e}")
            
            # Fallback: Try stored attention weights
            elif hasattr(module, 'last_attention_weights'):
                attn_weights = module.last_attention_weights
                if attn_weights is not None:
                    # attn_weights shape: (batch, num_heads, num_queries, num_keys)
                    attn_weights = attn_weights.mean(dim=1)  # Average over heads
                    
                    for token_idx in range(min(attn_weights.shape[3], mmdit_actual_prompt_length)):
                        token_attention = attn_weights[0, :, token_idx]  # (num_queries,)
                        
                        if token_attention.dtype == torch.bfloat16:
                            token_attention = token_attention.float()
                        
                        try:
                            attention_map = token_attention.reshape(h_img, w_img).cpu().numpy()
                            attention_maps[token_idx] = attention_map
                        except Exception as e:
                            pass
        
        # Register hook on MMDiT attention module
        try:
            mmdit_attn_module = pipe.transformer.get_submodule(hook_location)
            hook_handle = mmdit_attn_module.register_forward_hook(attention_hook)
            print(f"   ✅ Successfully hooked into {hook_location}")
        except Exception as e:
            print(f"   ⚠️  Could not hook into {hook_location}: {e}")
            return {}
        
        # Generate to capture attention
        try:
            _ = pipe(
                prompt=prompt,
                height=M,
                width=N,
                num_inference_steps=num_steps,
                guidance_scale=0.0,
                generator=generator,
            )
        finally:
            if 'hook_handle' in locals():
                hook_handle.remove()
        
        return attention_maps
    
    # Extract cross-attention maps
    try:
        mmdit_attention_maps_dict = extract_mmdit_cross_attention_maps(prompt, num_steps=1)
        
        if mmdit_attention_maps_dict:
            print(f"   ✅ Extracted attention maps for {len(mmdit_attention_maps_dict)} text tokens")
            
            # For each feature, find top activating token and create image-space sparse maps
            print("\n🔄 Converting Text Stream Features to Image Space...")
            print("   Strategy: For each feature, find top token → use its attention map")
            
            # Get top features (from earlier analysis)
            if 'top_features' in globals():
                features_to_process = top_features[:10]  # Process top 10 features
            else:
                # Compute top features if not available
                feature_means = text_sparse_maps[:mmdit_actual_prompt_length, :].mean(axis=0)
                features_to_process = feature_means.topk(10).indices.cpu().tolist()
            
            # Create text-guided sparse maps: (h_img, w_img, num_features)
            num_features = text_sparse_maps.shape[1]
            text_guided_sparse_maps = torch.zeros(h_img, w_img, num_features, 
                                                    device=text_sparse_maps.device, dtype=text_sparse_maps.dtype)
            
            # For each feature, find top activating token and use its attention map
            for feat_idx in range(num_features):
                # Get activations for this feature across all text tokens
                feature_activations = text_sparse_maps[:mmdit_actual_prompt_length, feat_idx]  # (actual_prompt_length,)
                
                # Find top activating token for this feature
                top_token_idx = feature_activations.argmax().item()
                top_token_activation = feature_activations[top_token_idx].item()
                
                # Get attention map for this top token
                if top_token_idx in mmdit_attention_maps_dict:
                    attention_map = torch.from_numpy(mmdit_attention_maps_dict[top_token_idx]).to(
                        device=text_sparse_maps.device, dtype=text_sparse_maps.dtype
                    )
                    
                    # Weight the attention map by the feature activation strength
                    # This creates a spatial map showing where this feature is active in image space
                    text_guided_sparse_maps[:, :, feat_idx] = attention_map * top_token_activation
            
            print(f"   ✅ Created text-guided sparse maps: {text_guided_sparse_maps.shape}")
            
            # Store for visualization
            text_guided_sparse_maps_available = True
            visualization_mode = "text_guided"  # Default to text-guided for MMDiT text stream
            
            # Print summary for top features
            print(f"\n📊 Top Token Analysis for Top {len(features_to_process)} Features:")
            print("="*80)
            for i, feat_idx in enumerate(features_to_process[:5]):  # Show first 5
                feature_activations = text_sparse_maps[:mmdit_actual_prompt_length, feat_idx]
                top_token_idx = feature_activations.argmax().item()
                top_token_activation = feature_activations[top_token_idx].item()
                token_text = mmdit_token_texts.get(top_token_idx, f"T{top_token_idx}")
                print(f"  Feature {feat_idx}: Top token = {top_token_idx} ({token_text}), Activation = {top_token_activation:.4f}")
            print("="*80)
            
        else:
            print("   ⚠️  Could not extract attention maps - using image-guided only")
            text_guided_sparse_maps_available = False
            visualization_mode = "image_guided"
            text_guided_sparse_maps = None
            
    except Exception as e:
        print(f"   ⚠️  Error extracting cross-attention maps: {str(e)}")
        traceback.print_exc()
        text_guided_sparse_maps_available = False
        visualization_mode = "image_guided"
        text_guided_sparse_maps = None
    
    print("\n✅ MMDiT Text Stream Analysis Complete")
    print("="*80)
    
else:
    # Not a text stream MMDiT SAE - initialize variables
    if not is_dit and stream == 1:
        print("ℹ️  Text stream detected but sparse_maps is not 2D - skipping cross-attention analysis")
    text_guided_sparse_maps_available = False
    visualization_mode = "image_guided"
    text_guided_sparse_maps = None
    if 'text_guided_sparse_maps' not in locals():
        text_guided_sparse_maps = None


## 8. Get Top Features

Identify the top activating features from the sparse maps for visualization and steering experiments.


In [ ]:
# Get top features
if len(sparse_maps.shape) == 3:
    # Spatial sparse maps: (h, w, features)
    # Compute mean across spatial dimensions
    feature_means = sparse_maps.mean(axis=(0, 1))
elif len(sparse_maps.shape) == 2:
    # Sequence sparse maps: (seq, features)
    # For text streams, we should filter padding tokens, but for simplicity, use all
    feature_means = sparse_maps.mean(axis=0)
else:
    raise ValueError(f"Unexpected sparse_maps shape: {sparse_maps.shape}")

# Get top 6 features
top_features = feature_means.topk(6).indices.cpu().tolist()
print("Top 6 features:", top_features)


## 9. Visualize Feature Activation Heatmaps

Visualize top features as heatmap overlays on the generated image. Supports both image-guided and text-guided visualization modes.


In [ ]:
# Visualize top 6 features heatmap overlay with example image
# Choose visualization mode: "image_guided" (default) or "text_guided" (for DiT layers only)
# Note: visualization_mode is set in the DiT text token analysis cell (Cell 6)
# To change mode, edit Cell 6 and set: visualization_mode = "text_guided"  (or "image_guided")
# For non-DiT layers or if text-guided maps are not available, it defaults to "image_guided"
visualization_mode = "text_guided"

# Determine which sparse maps to use
if visualization_mode == "text_guided" and text_guided_sparse_maps_available:
    sparse_maps_for_viz = text_guided_sparse_maps
    viz_label = "Text-Guided"
    print(f"📊 Using {viz_label} sparse maps for visualization")
else:
    sparse_maps_for_viz = sparse_maps
    viz_label = "Image-Guided"
    if visualization_mode == "text_guided":
        print(f"⚠️  Text-guided maps not available, using {viz_label} sparse maps")
    else:
        print(f"📊 Using {viz_label} sparse maps for visualization")

plt.figure(figsize=(15, 8))
for i, feature_idx in enumerate(top_features[:6]):
    plt.subplot(2, 3, i+1)
    heatmap_image = plot_image_heatmap(image, sparse_maps_for_viz, feature_idx)
    plt.imshow(heatmap_image)
    plt.title(f"Feature {feature_idx}\n({viz_label})")
    plt.axis("off")
plt.suptitle(f'Top 6 Feature Activation Heatmaps - {viz_label}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

prev_heatmap = heatmap_image
visualization_mode = "image_guided"

# Determine which sparse maps to use
if visualization_mode == "text_guided" and text_guided_sparse_maps_available:
    sparse_maps_for_viz = text_guided_sparse_maps
    viz_label = "Text-Guided"
    print(f"📊 Using {viz_label} sparse maps for visualization")
else:
    sparse_maps_for_viz = sparse_maps
    viz_label = "Image-Guided"
    if visualization_mode == "text_guided":
        print(f"⚠️  Text-guided maps not available, using {viz_label} sparse maps")
    else:
        print(f"📊 Using {viz_label} sparse maps for visualization")

plt.figure(figsize=(15, 8))
for i, feature_idx in enumerate(top_features[:6]):
    plt.subplot(2, 3, i+1)
    heatmap_image = plot_image_heatmap(image, sparse_maps_for_viz, feature_idx)
    plt.imshow(heatmap_image)
    plt.title(f"Feature {feature_idx}\n({viz_label})")
    plt.axis("off")
plt.suptitle(f'Top 6 Feature Activation Heatmaps - {viz_label}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# If it's MMDiT and stream is 1, we need to use the text stream
if not is_dit and stream == 1:
    sparse_maps = text_guided_sparse_maps

# Convert PIL.Image.Image objects to numpy arrays
heatmap_np = np.array(heatmap_image)
prev_heatmap_np = np.array(prev_heatmap)

diff = heatmap_np - prev_heatmap_np
# Quick statistics to check if heatmap and prev_heatmap are the same
print("Diff min:", diff.min(), "max:", diff.max(), "mean:", diff.mean(), "sum:", diff.sum())
if np.all(diff == 0):
    print("✅ Heatmap and prev_heatmap are identical (diff is all zeros).")
else:
    print("⚠️ Heatmap and prev_heatmap differ (diff is NOT all zeros).")

## 10. Empty-Prompt Intervention

Test feature effects in isolation by generating images with empty prompts and injecting features. This shows what each feature does without prompt influence.


In [ ]:
# Empty-prompt intervention (similar to SDXL example)
# This tests feature effects without prompt influence by using an empty prompt
def empty_prompt_intervention(sae, feature_idx, strength, hook_location, stream, is_dit=False, T_text=512):
    """
    Apply feature intervention with empty prompt to test feature effects in isolation.
    Similar to SDXL's empty_prompt_intervention.
    
    Args:
        sae: The SAE model
        feature_idx: Feature index to inject
        strength: Scaling factor for the feature
        hook_location: Where to hook in the transformer
        stream: Which stream to modify (0=query/image, 1=key/text)
        is_dit: Whether this is a DiT layer
        T_text: Number of text tokens (for computing strength with means)
    
    Returns:
        Generated image with feature intervention
    """
    def steering_hook(module, input, output):
        """Hook that replaces activations with feature direction"""
        if isinstance(output, tuple):
            query, key = output
            target = query if stream == 0 else key
            other = key if stream == 0 else query
            is_tuple = True
        else:
            target = output
            other = None
            is_tuple = False
        
        # Use replace_with_feature with mean activation as baseline
        # Similar to SDXL: strength * means[feature] * sae.k
        # For FLUX, we use means[feature_idx] from computed means
        mean_activation = means[feature_idx].item()
        # Scale by SAE's k parameter (top-k value) if available
        sae_k = getattr(sae, 'k', 1)  # Default to 1 if k not available
        feature_strength = strength * mean_activation * sae_k
        
        # Replace with feature direction
        modified_target = replace_with_feature(
            sae,
            feature_idx,
            feature_strength,
            target,
            stream=0  # Always 0 since we've already selected the target stream
        )
        
        if is_tuple:
            return (modified_target, other) if stream == 0 else (other, modified_target)
        else:
            return modified_target
    
    # Register hook
    hook_handle = pipe.transformer.get_submodule(hook_location).register_forward_hook(steering_hook)
    
    try:
        # Generate with empty prompt
        output = pipe(
            prompt="",  # Empty prompt to test feature in isolation
            height=M,
            width=N,
            num_inference_steps=test_timesteps,
            guidance_scale=0.0,
            generator=generator,
        )
        return output.images[0]
    finally:
        hook_handle.remove()

# Test empty-prompt intervention for top 6 features
print("Testing empty-prompt intervention for top 6 features...")
print("This shows what each feature does without prompt influence.")

# strengths_empty = [0.5, 1.0, 1.5, 2.0]  # Different strengths for empty prompt (typically smaller)
# strengths_empty = [i for i in np.arange(0, 5, 1)]
strengths_empty = [i for i in np.arange(0, 50, 10)]

fig, axes = plt.subplots(6, len(strengths_empty), figsize=(len(strengths_empty)*2, 12))  # 6 rows (features) x 3 columns (strengths)

for feat_idx, feature_idx in enumerate(top_features[:6]):
    print(f"Processing feature {feature_idx} ({feat_idx+1}/6)...")
    for strength_idx, strength in enumerate(strengths_empty):
        ax = axes[feat_idx, strength_idx]
        empty_image = empty_prompt_intervention(
            sae, feature_idx, strength, hook_location, stream, is_dit=is_dit, T_text=T_text
        )
        ax.imshow(empty_image)
        ax.set_xticks([])
        ax.set_yticks([])

        # Add column labels (strength values) on the left of each column
        if strength_idx== 0:
            ax.set_ylabel(f"Feature {feature_idx}", fontsize=10, fontweight='bold', va='center')
        # Add row labels (feature numbers) at the top of each row
        if feat_idx== 0:
            ax.set_title(f"Strength {strength}", fontsize=10, fontweight='bold', ha='center', va='center')

plt.tight_layout()
plt.show()
print("Done!")


## 11. Activation Modulation (Steering)

Apply feature steering with different methods and strengths. Supports multiple steering methods:
- **additive**: Direct feature direction addition (default, fastest)
- **surgery**: Full encode-decode surgery (most disruptive)
- **gytis_style**: Only boosts features in top-k (less disruptive)
- **error_preserving**: CLIP-style with reconstruction error preservation


In [ ]:
# Activation modulation with different strengths
# Methods: "additive" (default), "surgery", "gytis_style", "error_preserving"
def activation_modulation(sparse_maps, sae, feature_idx, strength, hook_location, stream, is_dit=False, sparse_maps_flat=None, T_img=None, T_text=None, method="additive"):
    """
    Apply activation modulation using different steering methods.
    
    Methods:
    - "additive": Direct feature direction addition (default, fastest)
    - "surgery": Full encode-decode surgery (most disruptive)
    - "gytis_style": Only boosts features in top-k (less disruptive)
    - "error_preserving": CLIP-style with reconstruction error preservation
    
    Args:
        sparse_maps: Original sparse maps (for getting spatial strength map)
        sae: The SAE model
        feature_idx: Feature index to modulate
        strength: Modulation strength
        hook_location: Where to hook in the transformer
        stream: Which stream to modify (0=query/image, 1=key/text)
        is_dit: Whether this is a DiT layer
        sparse_maps_flat: Full flat sparse maps for DiT layers
        T_img: Total image tokens (for DiT layers)
        T_text: Number of text tokens (default 512)
        method: Steering method ("additive", "surgery", "gytis_style", "error_preserving")
    """
    def steering_hook(module, input, output):
        """Hook that applies feature steering"""
        if isinstance(output, tuple):
            query, key = output
            target = query if stream == 0 else key
            other = key if stream == 0 else query
            is_tuple = True
        else:
            target = output
            other = None
            is_tuple = False
        
        # Get strength map from sparse_maps
        # Keep on same device and dtype as target tensor
        target_device = target.device
        target_dtype = target.dtype
        
        # Ensure SAE is on correct device/dtype
        sae_dtype = next(sae.parameters()).dtype
        sae_device = next(sae.parameters()).device
        
        # Handle DiT layers (fused stream: 512 text + T_img image tokens)
        if is_dit and sparse_maps_flat is not None:
            # Flatten target to get full sequence
            original_shape = target.shape
            target_flat = rearrange(target, "b ... d -> (b ...) d")
            num_tokens = target_flat.shape[0]
            
            # For DiT: sequence is [512 text tokens, T_img image tokens]
            t_img_local = T_img if T_img is not None else (num_tokens - T_text)

            # For text tokens (first 512), use zero (don't steer text)
            text_strength_flat = torch.zeros(T_text, device=target_device, dtype=target_dtype)

            # Get strength map for image tokens (next T_img tokens)
            image_strength_flat = sparse_maps_flat[T_text:T_text + t_img_local, feature_idx].to(device=target_device, dtype=target_dtype)

            # Concatenate: text first, then image
            strength_map_flat = torch.cat([text_strength_flat, image_strength_flat])

            # Convert target to SAE dtype/device for surgery methods
            if method != "additive":
                target_flat_sae = target_flat.to(dtype=sae_dtype, device=sae_device)
            else:
                target_flat_sae = target_flat
            
            # Apply steering based on method
            if method == "additive":
                # Original additive method
                modified_target_flat = add_feature_on_area(
                    sae,
                    feature_idx,
                    strength_map_flat * strength,
                    target_flat_sae,
                    stream=0
                )
            elif method == "surgery":
                # Full encode-decode surgery with per-token strength scaling
                # Surgery methods expect scalar strength, so we apply per-token
                # For efficiency, we can try to use tensor broadcasting if it works
                scaled_strengths = strength_map_flat * strength
                try:
                    # Try to use tensor strength (may work if surgery supports it)
                    encoded = sae.encode(target_flat_sae)
                    offset = torch.zeros_like(encoded)
                    offset[:, feature_idx] = scaled_strengths  # Per-token strength
                    modified_target_flat = sae.decode(encoded + offset)
                except:
                    # Fallback: apply per-token (slower but guaranteed to work)
                    modified_target_flat = torch.zeros_like(target_flat_sae)
                    for i in range(target_flat_sae.shape[0]):
                        token_strength = scaled_strengths[i].item()
                        modified_target_flat[i] = sae.surgery(
                            target_flat_sae[i:i+1],
                            k=feature_idx,
                            strength=token_strength
                        )[0]
            elif method == "gytis_style":
                # Gytis-style: only boosts features in top-k
                scaled_strengths = strength_map_flat * strength
                try:
                    # Try batch processing with tensor strengths
                    modified_target_flat = sae.surgery_gytis_style(
                        target_flat_sae,
                        k=feature_idx,
                        strength=scaled_strengths,
                        fallback_to_additive=True
                    )
                except:
                    # Fallback: apply per-token
                    modified_target_flat = torch.zeros_like(target_flat_sae)
                    for i in range(target_flat_sae.shape[0]):
                        token_strength = scaled_strengths[i].item()
                        modified_target_flat[i] = sae.surgery_gytis_style(
                            target_flat_sae[i:i+1],
                            k=feature_idx,
                            strength=token_strength,
                            fallback_to_additive=True
                        )[0]
            elif method == "error_preserving":
                # CLIP-style with reconstruction error preservation
                scaled_strengths = strength_map_flat * strength
                try:
                    # Try batch processing with tensor strengths
                    modified_target_flat = sae.surgery_with_error_preservation(
                        target_flat_sae,
                        k=feature_idx,
                        strength=scaled_strengths,
                        fallback_to_additive=True
                    )
                except:
                    # Fallback: apply per-token
                    modified_target_flat = torch.zeros_like(target_flat_sae)
                    for i in range(target_flat_sae.shape[0]):
                        token_strength = scaled_strengths[i].item()
                        modified_target_flat[i] = sae.surgery_with_error_preservation(
                            target_flat_sae[i:i+1],
                            k=feature_idx,
                            strength=token_strength,
                            fallback_to_additive=True
                        )[0]
            else:
                raise ValueError(f"Unknown method: {method}. Use 'additive', 'surgery', 'gytis_style', or 'error_preserving'")
            
            # Convert back to original dtype/device
            modified_target_flat = modified_target_flat.to(dtype=target_dtype, device=target_device)
            
            # Reshape back
            modified_target = modified_target_flat.reshape(original_shape)
        else:
            # MMDiT layers or non-DiT
            original_shape = target.shape
            target_flat = rearrange(target, "b ... d -> (b ...) d")
            
            if len(sparse_maps.shape) == 3:
                # Spatial: (h, w, features) -> flatten to match target_flat
                strength_map_flat = sparse_maps[:, :, feature_idx].flatten().to(device=target_device, dtype=target_dtype)
            elif len(sparse_maps.shape) == 2:
                # Sequence: (seq, features) - for text streams
                strength_map_flat = sparse_maps[:, feature_idx].to(device=target_device, dtype=target_dtype)
            else:
                # Scalar value - create tensor on target device
                strength_map_flat = torch.ones(target_flat.shape[0], device=target_device, dtype=target_dtype)
            
            # Ensure strength_map_flat matches target_flat length
            if strength_map_flat.shape[0] != target_flat.shape[0]:
                # If mismatch, use mean or broadcast
                if strength_map_flat.numel() == 1:
                    strength_map_flat = strength_map_flat.expand(target_flat.shape[0])
                else:
                    # Use mean as fallback
                    strength_map_flat = strength_map_flat.mean().expand(target_flat.shape[0])
            
            # Convert target to SAE dtype/device for surgery methods
            if method != "additive":
                target_flat_sae = target_flat.to(dtype=sae_dtype, device=sae_device)
            else:
                target_flat_sae = target_flat
            
            # Apply steering based on method
            if method == "additive":
                # Original additive method
                modified_target_flat = add_feature_on_area(
                    sae,
                    feature_idx,
                    strength_map_flat * strength,
                    target_flat_sae,
                    stream=0
                )
            elif method == "surgery":
                # Full encode-decode surgery with per-token strength scaling
                scaled_strengths = strength_map_flat * strength
                try:
                    # Try to use tensor strength (may work if surgery supports it)
                    encoded = sae.encode(target_flat_sae)
                    offset = torch.zeros_like(encoded)
                    offset[:, feature_idx] = scaled_strengths  # Per-token strength
                    modified_target_flat = sae.decode(encoded + offset)
                except:
                    # Fallback: apply per-token (slower but guaranteed to work)
                    modified_target_flat = torch.zeros_like(target_flat_sae)
                    for i in range(target_flat_sae.shape[0]):
                        token_strength = scaled_strengths[i].item()
                        modified_target_flat[i] = sae.surgery(
                            target_flat_sae[i:i+1],
                            k=feature_idx,
                            strength=token_strength
                        )[0]
            elif method == "gytis_style":
                # Gytis-style: only boosts features in top-k
                scaled_strengths = strength_map_flat * strength
                try:
                    # Try batch processing with tensor strengths
                    modified_target_flat = sae.surgery_gytis_style(
                        target_flat_sae,
                        k=feature_idx,
                        strength=scaled_strengths,
                        fallback_to_additive=True
                    )
                except:
                    # Fallback: apply per-token
                    modified_target_flat = torch.zeros_like(target_flat_sae)
                    for i in range(target_flat_sae.shape[0]):
                        token_strength = scaled_strengths[i].item()
                        modified_target_flat[i] = sae.surgery_gytis_style(
                            target_flat_sae[i:i+1],
                            k=feature_idx,
                            strength=token_strength,
                            fallback_to_additive=True
                        )[0]
            elif method == "error_preserving":
                # CLIP-style with reconstruction error preservation
                scaled_strengths = strength_map_flat * strength
                try:
                    # Try batch processing with tensor strengths
                    modified_target_flat = sae.surgery_with_error_preservation(
                        target_flat_sae,
                        k=feature_idx,
                        strength=scaled_strengths,
                        fallback_to_additive=True
                    )
                except:
                    # Fallback: apply per-token
                    modified_target_flat = torch.zeros_like(target_flat_sae)
                    for i in range(target_flat_sae.shape[0]):
                        token_strength = scaled_strengths[i].item()
                        modified_target_flat[i] = sae.surgery_with_error_preservation(
                            target_flat_sae[i:i+1],
                            k=feature_idx,
                            strength=token_strength,
                            fallback_to_additive=True
                        )[0]
            else:
                raise ValueError(f"Unknown method: {method}. Use 'additive', 'surgery', 'gytis_style', or 'error_preserving'")
            
            # Convert back to original dtype/device
            modified_target_flat = modified_target_flat.to(dtype=target_dtype, device=target_device)
            
            # Reshape back
            modified_target = modified_target_flat.reshape(original_shape)
        
        if is_tuple:
            return (modified_target, other) if stream == 0 else (other, modified_target)
        else:
            return modified_target
    
    # Register hook
    hook_handle = pipe.transformer.get_submodule(hook_location).register_forward_hook(steering_hook)
    
    try:
        # Generate with steering (progress bar already disabled globally)
        output = pipe(
            prompt=prompt,
            height=M,
            width=N,
            num_inference_steps=test_timesteps,
            guidance_scale=0.0,
            generator=generator,
        )
        return output.images[0]
    finally:
        hook_handle.remove()

# Test with different strengths for all top 6 features
strengths = [-10, -5, 0, 5, 10]
strengths = [i*5 for i in strengths]

# Steering method: "additive" (default), "surgery", "gytis_style", "error_preserving"
# - "additive": Direct feature direction addition (fastest, current default)
# - "surgery": Full encode-decode surgery (most disruptive, borrowed from toy.ipynb)
# - "gytis_style": Only boosts features in top-k (less disruptive)
# - "error_preserving": CLIP-style with reconstruction error preservation
# steering_method = "gytis_style"  # Change to "surgery", "gytis_style", or "error_preserving" to test different methods
steering_method = "additive"  # Change to "surgery", "gytis_style", or "error_preserving" to test different methods

print(f"Testing all top 6 features with strengths: {strengths}")
print(f"Using steering method: {steering_method}")

# Create a large figure for all features
fig, axes = plt.subplots(6, 5, figsize=(30, 25))  # 6 rows (features) x 5 columns (strengths)

for feat_idx, feature_idx in enumerate(top_features[:6]):
# for feat_idx, feature_idx in enumerate(top_features[:1]):
    print(f"Processing feature {feature_idx} ({feat_idx+1}/6)...")
    for strength_idx, strength in enumerate(strengths):
        ax = axes[feat_idx, strength_idx]
        modulated_image = activation_modulation(
            sparse_maps, sae, feature_idx, strength, hook_location, stream,
            is_dit=is_dit, sparse_maps_flat=sparse_maps_flat, 
            T_img=T_img, T_text=512, method=steering_method
        )
        ax.imshow(modulated_image)
        # Add row labels (feature numbers) at the very left of each row
        
        if strength_idx == 0:
            ax.set_ylabel(f"Feature {feature_idx}", fontsize=12, labelpad=40, va='center')
        # Add column labels (strength values) on the left of each column
        if feat_idx == 0:
            ax.set_title(f"Strength {strength}", fontsize=12, pad=10)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()
print("Done!")


In [ ]:
# import torch
# from diffusers import ZImagePipeline

# # 1. Load the pipeline
# # Use bfloat16 for optimal performance on supported GPUs
# pipe = ZImagePipeline.from_pretrained(
#     "Tongyi-MAI/Z-Image-Turbo",
#     torch_dtype=torch.bfloat16,
#     low_cpu_mem_usage=False,
# )
# pipe.to("cuda")

# # [Optional] Attention Backend
# # Diffusers uses SDPA by default. Switch to Flash Attention for better efficiency if supported:
# # pipe.transformer.set_attention_backend("flash")    # Enable Flash-Attention-2
# # pipe.transformer.set_attention_backend("_flash_3") # Enable Flash-Attention-3

# # [Optional] Model Compilation
# # Compiling the DiT model accelerates inference, but the first run will take longer to compile.
# # pipe.transformer.compile()

# # [Optional] CPU Offloading
# # Enable CPU offloading for memory-constrained devices.
# # pipe.enable_model_cpu_offload()

# # prompt = "Young Chinese woman in red Hanfu, intricate embroidery. Impeccable makeup, red floral forehead pattern. Elaborate high bun, golden phoenix headdress, red flowers, beads. Holds round folding fan with lady, trees, bird. Neon lightning-bolt lamp (⚡️), bright yellow glow, above extended left palm. Soft-lit outdoor night background, silhouetted tiered pagoda (西安大雁塔), blurred colorful distant lights."

# prompt = "A cinematic shot of a professor sloth wearing a green tuxedo at a BBQ party. Colour of suit: #7FFF00"
# # prompt = " A woman with a shirt and trousers, sitting on a chair, out of a restaurant named 'The Red Rabbit', with the logo of a rabbit with a pairs of deer horns. Colours: shirt: #191970, throusers: #7FFF00, chair: #C9A0DC, logo: #DC143C"
# prompt = "a woman with pink hair standing in a forest, holding a sign that says ‘Skipping flux blocks‘"

# # prompt = "a woman with pink hair standing in a dark room, holding a sign says `SAE‘. Behind her left is a blue light, right is a round red light. 4K. Photorealistic."
# # prompt = 'A faceted silver disco ball (front left) and a white ceramic mug (rear right) sitting on a white desk between two black gooseneck lamps. Right lamp turned on shining bright beam into the mug, glowing interior. Reflections sparkling on the disco ball. Left lamp turned off, dark room, high contrast, photorealistic, 4k.'
# # prompt = 'A faceted silver disco ball (front left) and a white ceramic mug (rear right) sitting on a white desk between two black gooseneck lamps. The left lamp is turned on, shining red soft light onto the disco ball, creating red reflections. The right lamp is turned on, shining a bright blue soft light directly into the mug. The white desk surface shows blended red and blue light patterns. Dark room, high contrast, photorealistic, 4k.'
# # prompt = 'Faceted silver disco ball front left, white ceramic mug rear right, on white desk between two black gooseneck lamps. Left lamp emitting soft red light onto ball. Right lamp emitting bright blue light into mug, glowing blue interior. Blended red blue light on desk surface. Dark room, high contrast, photorealistic, 4k.'
# # prompt = 'The capital of France is'
# # prompt = 'The capital of China is'

# # 2. Generate Image
# image = pipe(
#     prompt=prompt,
#     height=1024,
#     width=1024,
#     num_inference_steps=9,  # This actually results in 8 DiT forwards
#     guidance_scale=0.0,     # Guidance should be 0 for the Turbo models
#     generator=torch.Generator("cuda").manual_seed(42),
# ).images[0]

# # image.save("example.png")
# image


In [ ]:
# import torch
# from diffusers import Flux2Pipeline
# from diffusers.utils import load_image
# from huggingface_hub import get_token
# import requests
# import io

# repo_id = "diffusers/FLUX.2-dev-bnb-4bit" #quantized text-encoder and DiT. VAE still in bf16
# device = "cuda:0"
# torch_dtype = torch.bfloat16

# def remote_text_encoder(prompts):
#     response = requests.post(
#         "https://remote-text-encoder-flux-2.huggingface.co/predict",
#         json={"prompt": prompts},
#         headers={
#             "Authorization": f"Bearer {get_token()}",
#             "Content-Type": "application/json"
#         }
#     )
#     prompt_embeds = torch.load(io.BytesIO(response.content))

#     return prompt_embeds.to(device)
# print("Loading Flux2 pipeline...")
# pipe = Flux2Pipeline.from_pretrained(
#     repo_id, text_encoder=None, torch_dtype=torch_dtype
# ).to(device)
# print("Flux2 pipeline loaded.")

# # prompt = "A cinematic shot of a professor sloth wearing a tuxedo at a BBQ party."
# prompt = "a woman with pink hair standing in a forest, holding a sign that says ‘Skipping flux blocks‘"
# # prompt = "Realistic macro photograph of a hermit crab using a soda can as its shell, partially emerging from the can, captured with sharp detail and natural colors, on a sunlit beach with soft shadows and a shallow depth of field, with blurred ocean waves in the background. The can has the text `BFL Diffusers` on it and it has a color gradient that start with #FF5733 at the top and transitions to #33FF57 at the bottom."

# #cat_image = load_image("https://huggingface.co/spaces/zerogpu-aoti/FLUX.1-Kontext-Dev-fp8-dynamic/resolve/main/cat.png")
# image = pipe(
#     prompt_embeds=remote_text_encoder(prompt),
#     # height=512,
#     # width=512,
#     #image=[cat_image] #optional multi-image input
#     generator=torch.Generator(device=device).manual_seed(42),
#     num_inference_steps=28, #28 steps can be a good trade-off
#     guidance_scale=4,
# ).images[0]

# # image.save("flux2_output.png")
# image